# 📊 PIPELINE DE CONSOLIDATION ET DE NETTOYAGE DES DONNÉES
## Projet de Suivi des Dossiers et de la Veille Institutionnelle
---

### 1. Contexte Stratégique
Ce notebook implémente la couche **ETL (Extract, Transform, Load)** automatisée pour centraliser les flux de gestion des dossiers au sein de la Direction. L'objectif est d'assurer l'intégrité, l'alignement structurel et le nettoyage rigoureux des données brutes issues des services opérationnels avant leur modélisation et exploitation analytique dans **Power BI**.

### 2. Architecture des Flux d'Entrée
Les données proviennent de trois sources distinctes qui doivent être harmonisées verticalement :
* `Consolidation_Dossiers_Saisis.xlsx` ➔ Flux des dossiers traités.
* `DossiersNonSaisis.xlsx` ➔ Flux des dossiers en instance.
* `Liste des Dossiers Hors Délai.xlsx` ➔ Alertes de performance et ruptures de SLA.

In [2]:
import pandas as pd
from pathlib import Path

# 1. Résolution robuste des chemins de fichiers
root_dir = Path.cwd()
if not (root_dir / "data").exists():
    parent_dir = root_dir.parent
    if (parent_dir / "data").exists():
        root_dir = parent_dir
    else:
        raise FileNotFoundError(
            f"Impossible de trouver le dossier 'data' dans {root_dir} ou {parent_dir}. "
            "Exécutez le notebook depuis la racine du projet ou ajustez le chemin d'accès."
        )

data_dir = root_dir / "data"
file_saisis = data_dir / "Consolidation_Dossiers_Saisis.xlsx"
file_non_saisis = data_dir / "DossiersNonSaisis.xlsx"
file_hors_delai = data_dir / "Liste des Dossiers Hors Délai.xlsx"

for path in (file_saisis, file_non_saisis, file_hors_delai):
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")

print("Chargement des fichiers...")
df_saisis = pd.read_excel(file_saisis)
df_non_saisis = pd.read_excel(file_non_saisis)
df_hors_delai = pd.read_excel(file_hors_delai)

Chargement des fichiers...


### 3. Harmonisation Structurelle Sécurisée (Alignement des Schémas)

> ⚠️ **Règle métier critique :** Les fichiers sources issus des extractions de la plateforme Web souffrent de variations structurelles (colonnes masquées, métadonnées locales).

Pour garantir une fusion verticale sans décalage de lignes, l'algorithme applique un **bridage strict aux 8 premières colonnes clés** et force l'application du schéma directeur suivant :

| Ordre | Nom du Champ | Type Cible | Rôle Métier |
| :---: | :--- | :--- | :--- |
| **1** | `Matricule` | Alphanumérique | Identifiant unique de l'agent instructeur |
| **2** | `Nom Agent` | Texte | Identité de l'opérateur |
| **3** | `No Bordereau`| Alphanumérique | Référence du lot de transmission |
| **4** | `No Dossier`  | Alphanumérique | Clé primaire unique du dossier |
| **5** | `Type Doss`   | Texte (Catégoriel)| Nature de la procédure |
| **6** | `Montant`     | Décimal | Impact financier de l'acte |
| **7** | `Date Arriv`  | Date / Heure | Point de départ de l'instruction |
| **8** | `Durée Dossier`| Entier | Délai de traitement (en jours) |

*En cas de non-respect de ce schéma par un fichier source, l'exécution est immédiatement interrompue (`ValueError`) pour empêcher la corruption du rapport final.*

In [3]:
# =====================================================================
# NETTOYAGE ET HARMONISATION SÉCURISÉE
# =====================================================================
colonnes_reference = ["Matricule", "Nom Agent", "No Bordereau", "No Dossier", "Type Doss", "Montant", "Date Arriv", "Durée Dossier"]

def harmoniser_tableau(df, nom_fichier):
    if len(df.columns) >= 8:
        df = df.iloc[:, :8]
    else:
        raise ValueError(f"Le fichier {nom_fichier} a moins de 8 colonnes.")
    
    df.columns = colonnes_reference
    return df

df_saisis = harmoniser_tableau(df_saisis, file_saisis)
df_non_saisis = harmoniser_tableau(df_non_saisis, file_non_saisis)
df_hors_delai = harmoniser_tableau(df_hors_delai, file_hors_delai)
# =====================================================================

### 4. Traçabilité des Flux (Enrichissement Métier)
Afin de ne pas perdre la traçabilité de l'origine des données après la fusion, une colonne catégorielle `Statut` est injectée dynamiquement dans chaque sous-ensemble. Ce marqueur permettra à Power BI de segmenter instantanément les indicateurs clés de performance (KPI).

In [4]:
# 2. Injection propre du Statut
df_saisis['Statut'] = 'Saisi'
df_non_saisis['Statut'] = 'Non Saisi'
df_hors_delai['Statut'] = 'Hors Délai'

# 3. Fusion verticale
print("Fusion alignée en cours...")
master_table = pd.concat([df_saisis, df_non_saisis, df_hors_delai], ignore_index=True)

# 4. Nettoyage des lignes de titres parasites
master_table = master_table[master_table['Matricule'].astype(str).str.contains('Matricule|Exportation') == False]
master_table = master_table.dropna(how='all', subset=["No Dossier"])

Fusion alignée en cours...


### 5. Résolution des Anomalies de Types et Nettoyage des Lignes Parasites

Cette étape traite deux anomalies majeures identifiées lors des audits de données précédents :
1.  **Lignes d'en-tête répétitives :** Élimination des scories d'exportation (lignes contenant des labels textuels répétés ou des dossiers fantômes sans identifiant).
2.  **Corruption des types numériques (`Durée Dossier` & `Montant`) :** Les anomalies textuelles ou les cellules vides sont converties de manière sécurisée en `NaN` (`errors='coerce'`), puis normalisées à `0`.

*Cette standardisation élimine radicalement les erreurs de conversion de type lors du rafraîchissement automatique dans Power BI.*

In [5]:
# =====================================================================
# CORRECTION DU BUG DE TYPE (DURÉE DOSSIER)
# =====================================================================
# On force la colonne Durée Dossier à devenir numérique. 
# 'coerce' va transformer automatiquement toutes les erreurs/textes/dates parasites en 'NaN' (vide numérique)
master_table['Durée Dossier'] = pd.to_numeric(master_table['Durée Dossier'], errors='coerce')

# On remplace les NaN par 0 pour que Power BI reçoive un vrai chiffre propre
master_table['Durée Dossier'] = master_table['Durée Dossier'].fillna(0).astype(int)

# On fait pareil pour Montant au cas où
master_table['Montant'] = pd.to_numeric(master_table['Montant'], errors='coerce').fillna(0)
# =====================================================================

### 6. Génération du Livrable et Optimisation du Stockage
Le dataset consolidé est exporté au format **CSV optimisé**.

* **Encodage :** `utf-8-sig` (Garantit la parfaite conservation des accents et caractères spécifiques dans Excel et Power BI Desktop sous Windows).
* **Séparateur :** Point-virgule `;` (Conforme aux paramètres régionaux des systèmes de l'administration).
* **Destination :** `Master_Table_Ministere.csv` ➔ Prêt pour la connexion directe au modèle Power BI.

In [ ]:
# 5. Exportation en CSV optimisé
output_file = "../data/Master_Table_Ministere.csv"
master_table.to_csv(output_file, index=False, encoding='utf-8-sig', sep=';')

print(f"✅ Terminé ! Version corrigée sans erreurs générée : {output_file}")

✅ Terminé ! Version corrigée sans erreurs générée : Master_Table_Ministere.csv
